# CWRU bearing signals: healthy vs inner-race fault

We compare `97.mat` (normal, 0 HP) and `105.mat` (12k drive-end, 0.007 inch inner-race fault, 0 HP). The CWRU pages list the normal 0 HP speed as about 1797 rpm and describe drive-end recordings at 12 kHz and 48 kHz. The normal-baseline table does not identify the sample rate of `97.mat` per file, so these first plots use sample index. Verify that file's rate before using seconds or interpreting frequencies.

Sources: [CWRU normal baseline](https://engineering.case.edu/bearingdatacenter/normal-baseline-data), [CWRU 12k drive-end data](https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data), [CWRU download notes](https://engineering.case.edu/bearingdatacenter/download-data-file).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

from motorsense_ml.features import crest_factor, kurtosis, peak, rms

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ml" / "data").is_dir())
DATA_DIR = ROOT / "ml" / "data"
FILES = {"healthy": DATA_DIR / "97.mat", "inner-race fault": DATA_DIR / "105.mat"}
WINDOW_SAMPLES = 4096

signals = {}
for label, path in FILES.items():
    mat = loadmat(path)
    keys = [key for key in mat if key.endswith("_DE_time")]
    if len(keys) != 1:
        raise ValueError(f"Expected one *_DE_time variable in {path.name}; found {keys}")
    signal = np.asarray(mat[keys[0]]).reshape(-1)
    signals[label] = (keys[0], signal)
    print(f"{label}: {keys[0]}, {signal.size:,} samples")

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(12, 6), sharex=True, constrained_layout=True)
for ax, (label, (_, signal)) in zip(axes, signals.items(), strict=False):
    shown = signal[:WINDOW_SAMPLES]
    ax.plot(np.arange(shown.size), shown, linewidth=0.7)
    ax.set_title(label)
    ax.set_ylabel("Acceleration (recorded units)")
axes[-1].set_xlabel("Sample index (sampling rate not applied)")
plt.show()

In [ ]:
feature_rows = []
for label, (_, signal) in signals.items():
    window = signal[:WINDOW_SAMPLES]
    feature_rows.append(
        {
            "condition": label,
            "rms": rms(window),
            "peak": peak(window),
            "crest_factor": crest_factor(window),
            "kurtosis": kurtosis(window),
        }
    )

for row in feature_rows:
    print(row)

## Check the sample rate using shaft speed

The normal recording's shaft speed is about 1797 rpm, or 29.95 Hz. Since the sampling rate is what we're checking, first view the spectrum in cycles per sample. If the selected peak is the `h`th shaft harmonic, then `fs = h × 29.95 / f_peak`. Compare the line and its harmonics against the 12 kHz and 48 kHz candidate locations. This is a consistency check: don't label the largest low-frequency peak as 1× without checking its order.

In [ ]:
_, normal = signals["healthy"]
x = normal[: 2**17]
x = x - x.mean()
N = len(x)
window = np.hanning(N)
xw = x * window
spectrum = np.abs(np.fft.rfft(xw))
freq_cycles_per_sample = np.fft.rfftfreq(N, d=1.0)

shaft_hz = 1797 / 60
search = (freq_cycles_per_sample >= 2 / N) & (freq_cycles_per_sample <= 0.01)
peak_bin = np.flatnonzero(search)[np.argmax(spectrum[search])]
f_peak = freq_cycles_per_sample[peak_bin]
print(f"Largest low-frequency peak: bin {peak_bin}, {f_peak:.6f} cycles/sample")
for order in (1, 2, 3):
    print(f"If this is the {order}× shaft harmonic: fs ≈ {order * shaft_hz / f_peak:.0f} Hz")
for candidate_fs in (12_000, 48_000):
    peak_order = f_peak * candidate_fs / shaft_hz
    print(f"At fs={candidate_fs} Hz, selected peak is {peak_order:.1f}× shaft speed")
    for harmonic in (1, 2, 3):
        expected = harmonic * shaft_hz / candidate_fs
        center_bin = round(expected * N)
        lo = max(2, center_bin - 5)
        hi = min(len(spectrum), center_bin + 6)
        local_bin = lo + np.argmax(spectrum[lo:hi])
        print(
            f"  {harmonic}× expected {expected:.6f}: local peak "
            f"{freq_cycles_per_sample[local_bin]:.6f} cycles/sample"
        )

fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.semilogy(freq_cycles_per_sample[2:], spectrum[2:], linewidth=0.8)
ax.axvline(f_peak, color="black", linestyle="-", label="selected peak")
for candidate_fs, color in ((12_000, "tab:blue"), (48_000, "tab:orange")):
    for harmonic in (1, 2, 3):
        ax.axvline(
            harmonic * shaft_hz / candidate_fs,
            color=color,
            linestyle="--",
            alpha=0.65,
            label=f"{harmonic}× shaft if fs={candidate_fs // 1000} kHz" if harmonic == 1 else None,
        )
ax.set_xlim(0, 0.01)
ax.set_xlabel("Frequency (cycles/sample)")
ax.set_ylabel("Magnitude (windowed FFT)")
ax.set_title("97.mat normal drive-end spectrum")
ax.legend()
plt.show()